In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = Path(r"./saida/feature_extraction")
DATA_DIR_LABEL = Path(r"./rotulo")
DATA_DIR_OUT = Path(r"./saida/feature_extraction_labeled")
LABELS_PATH = DATA_DIR_LABEL / "labels_falha.csv"
AUG_PREFIXES = ("aug_brightness_", "aug_combined_", "aug_flip_", "aug_time_warp_")

In [3]:
def get_parent_video_id(video_id: str) -> str:
    """Se for vídeo aumentado, retorna o id do pai; senão retorna ele mesmo."""
    for p in AUG_PREFIXES:
        if video_id.startswith(p):
            return video_id[len(p):]
    return video_id

In [ ]:
labels = pd.read_csv(LABELS_PATH)

# lê *_reps.csv e adiciona video_id e parent_video_id
all_reps = []
for f in DATA_DIR.glob("*_reps.csv"):
    video_id = f.name.replace("_reps.csv", "")
    r = pd.read_csv(f)
    r["video_id"] = video_id
    r["parent_video_id"] = get_parent_video_id(video_id)
    all_reps.append(r)

reps = pd.concat(all_reps, ignore_index=True)

# 1) tenta label direto (se você tiver anotado algum aug_ manualmente)
df = reps.merge(labels, on="video_id", how="left")

# 2) se não achou, herda do pai
labels_parent = labels.rename(columns={
    "video_id": "parent_video_id",
    "fail_rep_id": "fail_rep_id_parent"
})
df = df.merge(labels_parent, on="parent_video_id", how="left")

# 3) escolhe o label final (direto > pai)
df["fail_rep_id_final"] = df["fail_rep_id"].combine_first(df["fail_rep_id_parent"])

# 4) converte de forma robusta (mantém 0-based!)
df["fail_rep_id_final"] = pd.to_numeric(df["fail_rep_id_final"], errors="coerce")

# 5) gera label por repetição: fadiga a partir da falha (inclusive)
df["fatigue_label"] = np.where(
    df["fail_rep_id_final"].notna() & (df["rep_id"] >= df["fail_rep_id_final"]),
    1,
    0
).astype(int)

out_path = DATA_DIR_OUT / "reps_labeled_all.csv"
df.to_csv(out_path, index=False)

: 